In [1]:
print("hello world")

hello world


In [2]:
import sys

print(sys.version)

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


In [3]:
!pip install google-api-python-client
!pip install google-auth
!pip install google-auth-oauthlib
!pip install google-auth-httplib2
!pip install pandas

In [4]:
from google.colab import auth

auth.authenticate_user()

print("Authentication Successful")

Authentication Successful


In [5]:
from googleapiclient.discovery import build

service = build("drive", "v3")

print("Google Drive Connected Successfully")

Google Drive Connected Successfully


In [6]:
results = service.files().list(
    pageSize=5,
    fields="files(id,name)"
).execute()

files = results.get("files", [])

for file in files:
    print(file["name"])

CSV_Automation.ipynb
practice2.sql
practice1.sql
CSV_Row_Splitter.ipynb
Overall_Practice_1.sql


In [7]:
# Input Folder ID
INPUT_FOLDER_ID = "YOUR INPUT FOLDER ID"


def get_latest_csv(folder_id):

    query = (
        f"'{folder_id}' in parents "
        "and mimeType='text/csv' "
        "and trashed=false"
    )

    results = service.files().list(
        q=query,
        orderBy="modifiedTime desc",
        pageSize=1,
        fields="files(id,name,modifiedTime)"
    ).execute()

    files = results.get("files", [])

    if len(files) == 0:
        raise Exception("No CSV files found.")

    latest_file = files[0]

    print("Latest CSV Found")
    print("----------------------------")
    print("File Name :", latest_file["name"])
    print("File ID   :", latest_file["id"])
    print("Modified  :", latest_file["modifiedTime"])

    return latest_file["id"]

In [8]:
latest_file_id = get_latest_csv(INPUT_FOLDER_ID)

print("\nLatest File ID :", latest_file_id)

Latest CSV Found
----------------------------
File Name : sample_data_20x10.csv
File ID   : 1NvtMx6jpOAI-yR2GDDcqsbjM0vHmpwA7
Modified  : 2026-10-05T05:28:42.600Z

Latest File ID : 1NvtMx6jpOAI-yR2GDDcqsbjM0vHmpwA7


In [9]:
import os
import io

from googleapiclient.http import MediaIoBaseDownload

# Create download folder
DOWNLOAD_FOLDER = "/content/downloads"

os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)


def download_csv(file_id):

    # Get file name
    file = service.files().get(
        fileId=file_id,
        fields="name"
    ).execute()

    file_name = file["name"]

    request = service.files().get_media(
        fileId=file_id
    )

    file_path = os.path.join(
        DOWNLOAD_FOLDER,
        file_name
    )

    fh = io.FileIO(file_path, "wb")

    downloader = MediaIoBaseDownload(
        fh,
        request
    )

    done = False

    while not done:

        status, done = downloader.next_chunk()

        print(
            f"Downloading : {int(status.progress()*100)}%"
        )

    print("Download Completed")

    return file_path

In [10]:
csv_path = download_csv(latest_file_id)

print("\nDownloaded File:")
print(csv_path)

Download Completed

Downloaded File:
/content/downloads/sample_data_20x10.csv


In [11]:
import os

print(os.listdir("/content/downloads"))

['sample_data_20x10.csv']


In [12]:
import pandas as pd

# Create split folder
SPLIT_FOLDER = "/content/split_output"

os.makedirs(SPLIT_FOLDER, exist_ok=True)


def find_id_column(df):

    possible_names = [
        "cust_id",
        "customer_id",
        "id",
        "ID",
        "CustID",
        "CustomerID",
        "customerId"
    ]

    for col in possible_names:
        if col in df.columns:
            return col

    for col in df.columns:
        if df[col].is_unique:
            return col

    raise Exception("No unique ID column found.")

In [13]:
def split_csv(csv_path):

    df = pd.read_csv(csv_path)

    id_column = find_id_column(df)

    print(f"ID Column : {id_column}")

    for _, row in df.iterrows():

        file_name = f"{row[id_column]}.csv"

        output_path = os.path.join(
            SPLIT_FOLDER,
            file_name
        )

        row.to_frame().T.to_csv(
            output_path,
            index=False
        )

        print(f"Created : {file_name}")

    print("\nAll files created successfully.")

In [14]:
split_csv(csv_path)

ID Column : Employee_ID
Created : EMP1001.csv
Created : EMP1002.csv
Created : EMP1003.csv
Created : EMP1004.csv
Created : EMP1005.csv
Created : EMP1006.csv
Created : EMP1007.csv
Created : EMP1008.csv
Created : EMP1009.csv
Created : EMP1010.csv
Created : EMP1011.csv
Created : EMP1012.csv
Created : EMP1013.csv
Created : EMP1014.csv
Created : EMP1015.csv
Created : EMP1016.csv
Created : EMP1017.csv
Created : EMP1018.csv
Created : EMP1019.csv
Created : EMP1020.csv

All files created successfully.


In [15]:
import os

print(os.listdir(SPLIT_FOLDER))

['EMP1016.csv', 'EMP1009.csv', 'EMP1011.csv', 'EMP1006.csv', 'EMP1002.csv', 'EMP1018.csv', 'EMP1019.csv', 'EMP1017.csv', 'EMP1003.csv', 'EMP1010.csv', 'EMP1004.csv', 'EMP1008.csv', 'EMP1013.csv', 'EMP1020.csv', 'EMP1015.csv', 'EMP1007.csv', 'EMP1014.csv', 'EMP1012.csv', 'EMP1005.csv', 'EMP1001.csv']


In [16]:
OUTPUT_FOLDER_ID = "YOUR OUTPUT FOLDER ID"

In [17]:
from googleapiclient.http import MediaFileUpload


def upload_split_files():

    files = os.listdir(SPLIT_FOLDER)

    print("Total Files :", len(files))


    for file_name in files:

        file_path = os.path.join(
            SPLIT_FOLDER,
            file_name
        )


        file_metadata = {
            "name": file_name,
            "parents": [OUTPUT_FOLDER_ID]
        }


        media = MediaFileUpload(
            file_path,
            mimetype="text/csv"
        )


        service.files().create(
            body=file_metadata,
            media_body=media,
            fields="id"
        ).execute()


        print(f"Uploaded : {file_name}")


    print("\nAll Files Uploaded Successfully")

In [18]:
upload_split_files()

Total Files : 20
Uploaded : EMP1016.csv
Uploaded : EMP1009.csv
Uploaded : EMP1011.csv
Uploaded : EMP1006.csv
Uploaded : EMP1002.csv
Uploaded : EMP1018.csv
Uploaded : EMP1019.csv
Uploaded : EMP1017.csv
Uploaded : EMP1003.csv
Uploaded : EMP1010.csv
Uploaded : EMP1004.csv
Uploaded : EMP1008.csv
Uploaded : EMP1013.csv
Uploaded : EMP1020.csv
Uploaded : EMP1015.csv
Uploaded : EMP1007.csv
Uploaded : EMP1014.csv
Uploaded : EMP1012.csv
Uploaded : EMP1005.csv
Uploaded : EMP1001.csv

All Files Uploaded Successfully


In [19]:
CLIENT_FOLDER_ID = "ENTER CLIENT FILDER ID"


folder = service.files().get(
    fileId=CLIENT_FOLDER_ID,
    fields="name"
).execute()


print(folder["name"])

new_upload_folder


In [20]:
CLIENT_FOLDER_ID = "ENTER CLIENT FILDER ID"


def transfer_files():

    query = (
        f"'{OUTPUT_FOLDER_ID}' in parents "
        "and trashed=false"
    )


    results = service.files().list(
        q=query,
        fields="files(id,name)"
    ).execute()


    files = results.get("files", [])


    print("Total Files :", len(files))


    for file in files:

        file_metadata = {
            "name": file["name"],
            "parents": [CLIENT_FOLDER_ID]
        }


        service.files().copy(
            fileId=file["id"],
            body=file_metadata
        ).execute()


        print("Transferred :", file["name"])


    print("\nTransfer Completed Successfully")

In [21]:
transfer_files()

Total Files : 63
Transferred : EMP1001.csv
Transferred : EMP1005.csv
Transferred : EMP1012.csv
Transferred : EMP1014.csv
Transferred : EMP1007.csv
Transferred : EMP1015.csv
Transferred : EMP1020.csv
Transferred : EMP1013.csv
Transferred : EMP1008.csv
Transferred : EMP1004.csv
Transferred : EMP1010.csv
Transferred : EMP1003.csv
Transferred : EMP1017.csv
Transferred : EMP1019.csv
Transferred : EMP1018.csv
Transferred : EMP1002.csv
Transferred : EMP1006.csv
Transferred : EMP1011.csv
Transferred : EMP1009.csv
Transferred : EMP1016.csv
Transferred : 101.csv
Transferred : 102.csv
Transferred : 103.csv
Transferred : EMP1008.csv
Transferred : EMP1004.csv
Transferred : EMP1010.csv
Transferred : EMP1016.csv
Transferred : EMP1001.csv
Transferred : EMP1018.csv
Transferred : EMP1002.csv
Transferred : EMP1003.csv
Transferred : EMP1012.csv
Transferred : EMP1017.csv
Transferred : EMP1020.csv
Transferred : EMP1019.csv
Transferred : EMP1006.csv
Transferred : EMP1009.csv
Transferred : EMP1014.csv
Transfe